In [1]:
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
from multiprocessing import shared_memory
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)

Running with 7 CPU cores | 15.8GB total RAM (7.2GB available) | pyarrow 24.0.0


Variables

In [2]:
%run 0_variables.ipynb

Feature ranking

In [3]:
import time as _time
_t0 = _time.perf_counter()
print(f"Loading features parquet: {os.environ['FEATURES_PATH']}", flush=True)
features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
    ('SETTLEMENTDATE', '>=', pd.Timestamp(os.environ["FEATURE_DATASET_START"])),
    ('SETTLEMENTDATE', '<=', pd.Timestamp(os.environ["FEATIRE_DATASET_END"])),
])
features = features.drop(columns=[c for c in features.columns if c in set(os.environ["TARGET_COLS"].split(","))])
features = features.loc[:pd.Timestamp(os.environ["FEATIRE_DATASET_END"])]
print(f"  loaded features: shape={features.shape} in {_time.perf_counter() - _t0:.1f}s", flush=True)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_t0 = _time.perf_counter()
print(f"Loading aggregated targets parquet for target={os.environ['TARGET']}, dataset={_stem}", flush=True)
future_prediction_targets_agg = pd.read_parquet(
    resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_targets_agg_{_stem}.parquet")
)
print(f"  loaded targets: shape={future_prediction_targets_agg.shape} in {_time.perf_counter() - _t0:.1f}s", flush=True)

"""
Rank features

MI ranking: measures mutual information — a non-linear, information-theoretic score. It answers "does knowing this feature reduce uncertainty about the target?",
capturing non-linear dependencies too. Targets are passed in pre-aggregated to output_resolution_minutes,
so MI is computed directly at the output resolution.
"""

# Worker-side cache: each loky worker reuses its SharedMemory attachment + ndarray view
# across the many tasks it processes (avoids re-attaching/re-wrapping per task).
_WORKER_SHM_CACHE = {}

def _attach_shared(name, shape, dtype):
    cached = _WORKER_SHM_CACHE.get(name)
    if cached is not None:
        return cached[1]
    shm = shared_memory.SharedMemory(name=name)
    arr = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
    _WORKER_SHM_CACHE[name] = (shm, arr)  # keep shm alive for the worker's lifetime
    return arr

def rank_features(features:pd.DataFrame,
    future_prediction_targets_agg: pd.DataFrame,
    feature_selection_subsample_start: pd.Timestamp,
    feature_selection_subsample_end: pd.Timestamp,
    feature_selection_subsample_amount: int,
    output_resolution_minutes: int,
):
    feature_cols = list(features.columns)
    target_cols_agg = list(future_prediction_targets_agg.columns)
    n_buckets = len(target_cols_agg)

    def _filter_data_for_feature_time_range_subset():
        print(f"  filtering to subsample window [{feature_selection_subsample_start} .. {feature_selection_subsample_end}]", flush=True)
        features_subset = features.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        targets_subset = future_prediction_targets_agg.loc[feature_selection_subsample_start:feature_selection_subsample_end]
        shared_index = features_subset.index.intersection(targets_subset.index)
        print(f"    window rows: features={len(features_subset):,} targets={len(targets_subset):,} shared={len(shared_index):,}", flush=True)
        return features_subset.loc[shared_index], targets_subset.loc[shared_index]

    features_subset, targets_subset = _filter_data_for_feature_time_range_subset()

    # Detect discrete-looking columns BEFORE the float cast, so dtype info is preserved.
    # Integer dtypes are treated as discrete; small-cardinality float columns (<=32 uniques,
    # all integral values) are also flagged. Discrete-feature MI uses a much faster code path
    # in sklearn (no per-feature kNN noise injection / joint-space search).
    print(f"  detecting discrete features", flush=True)
    _t = _time.perf_counter()
    _discrete_mask = np.zeros(features_subset.shape[1], dtype=bool)
    for _i, _col in enumerate(feature_cols):
        _s = features_subset[_col]
        if _s.dtype.kind in ("i", "u", "b"):
            _discrete_mask[_i] = True
            continue
        if _s.dtype.kind == "f":
            _vals = _s.to_numpy()
            # cheap check: small unique count AND all integral
            if _vals.size and np.all(np.isfinite(_vals)):
                if np.unique(_vals).size <= 32 and np.all(_vals == np.floor(_vals)):
                    _discrete_mask[_i] = True
    print(
        f"    discrete features: {_discrete_mask.sum()} / {len(feature_cols)} "
        f"in {_time.perf_counter() - _t:.1f}s",
        flush=True,
    )

    # Avoid an unnecessary float32 copy when the underlying block is already float32 contiguous.
    print(f"  preparing feature ndarray (shape={features_subset.shape})", flush=True)
    _t = _time.perf_counter()
    _arr = features_subset.to_numpy(dtype=np.float32, copy=False)
    if not _arr.flags["C_CONTIGUOUS"]:
        _arr = np.ascontiguousarray(_arr)
    features_subset = _arr
    targets_subset = targets_subset.reset_index(drop=True)
    print(
        f"    ready in {_time.perf_counter() - _t:.1f}s | feature matrix "
        f"{features_subset.nbytes / 1024**2:.0f}MB | contiguous={features_subset.flags['C_CONTIGUOUS']}",
        flush=True,
    )

    def _subsample_features():
        seed = np.random.default_rng(42)
        n_samples = min(feature_selection_subsample_amount, len(features_subset))
        print(f"  subsampling {n_samples:,} of {len(features_subset):,} rows", flush=True)
        subsample_index = seed.choice(len(features_subset), size=n_samples, replace=False)
        subsample_index.sort()
        return features_subset[subsample_index], targets_subset.iloc[subsample_index]

    X_subsample, y_subsample = _subsample_features()

    def _mutual_information_scoring():
        aggregated_target_matrix = y_subsample[target_cols_agg].values.astype(np.float32)
        n_features = X_subsample.shape[1]
        n_subsample_rows = X_subsample.shape[0]

        # Chunk features so total tasks = n_buckets * n_chunks >> _NCPU. This saturates
        # all cores even when n_buckets < _NCPU and lets the scheduler load-balance
        # cheap discrete columns against expensive continuous ones.
        _target_tasks = max(_NCPU * 4, n_buckets * 2)
        _n_chunks = max(1, min(n_features, _target_tasks // max(1, n_buckets)))
        _chunk_size = max(1, (n_features + _n_chunks - 1) // _n_chunks)
        _chunks = [
            (start, min(start + _chunk_size, n_features))
            for start in range(0, n_features, _chunk_size)
        ]
        _n_tasks = len(_chunks) * n_buckets

        print(
            f"  MI scoring: {n_features} features × {n_buckets} horizons ({output_resolution_minutes}-min) "
            f"| subsample n={n_subsample_rows:,} | {_n_tasks} tasks ({len(_chunks)} feature chunks × {n_buckets} buckets) "
            f"across {_NCPU} CPUs | ~{_MEMORY_PER_WORKER / 1024**2:.0f}MB per worker",
            flush=True,
        )

        shm_X = None
        shm_y = None
        try:
            # Place arrays in shared memory so workers receive only a name string, not a pickled copy
            print(f"  allocating shared memory: X={X_subsample.nbytes / 1024**2:.0f}MB, y={aggregated_target_matrix.nbytes / 1024**2:.0f}MB", flush=True)
            _t = _time.perf_counter()
            shm_X = shared_memory.SharedMemory(create=True, size=X_subsample.nbytes)
            shm_y = shared_memory.SharedMemory(create=True, size=aggregated_target_matrix.nbytes)
            np.copyto(np.ndarray(X_subsample.shape, dtype=np.float32, buffer=shm_X.buf), X_subsample)
            np.copyto(np.ndarray(aggregated_target_matrix.shape, dtype=np.float32, buffer=shm_y.buf), aggregated_target_matrix)
            X_shape, y_shape = X_subsample.shape, aggregated_target_matrix.shape
            shm_X_name, shm_y_name = shm_X.name, shm_y.name
            print(f"    shared memory ready in {_time.perf_counter() - _t:.1f}s", flush=True)

            def _score_chunk(bucket_idx, col_start, col_end, chunk_discrete_mask):
                # Reuse worker-cached attachments across tasks (see _attach_shared).
                X = _attach_shared(shm_X_name, X_shape, np.float32)
                y = _attach_shared(shm_y_name, y_shape, np.float32)
                X_chunk = X[:, col_start:col_end]
                # Pass discrete_features as a boolean mask so sklearn uses the fast
                # discrete branch where applicable.
                result = mutual_info_regression(
                    X_chunk,
                    y[:, bucket_idx],
                    discrete_features=chunk_discrete_mask,
                    n_neighbors=3,
                    random_state=42,
                )
                return bucket_idx, col_start, col_end, result

            print(
                f"  spawning {_NCPU} loky workers and dispatching {_n_tasks} tasks "
                f"(first results may take ~30s while workers warm up)",
                flush=True,
            )
            _t_dispatch = _time.perf_counter()
            gen = Parallel(n_jobs=_NCPU, backend="loky", batch_size=1, return_as="generator_unordered", verbose=10)(
                delayed(_score_chunk)(bucket_idx, cs, ce, _discrete_mask[cs:ce])
                for bucket_idx in range(n_buckets)
                for (cs, ce) in _chunks
            )
            scores = np.empty((n_features, n_buckets))
            _first = True
            for bucket_idx, cs, ce, chunk_scores in tqdm(gen, total=_n_tasks, desc="MI scoring", leave=True):
                if _first:
                    print(f"    first worker result received after {_time.perf_counter() - _t_dispatch:.1f}s", flush=True)
                    _first = False
                scores[cs:ce, bucket_idx] = chunk_scores
        finally:
            if shm_X is not None:
                shm_X.close(); shm_X.unlink()
            if shm_y is not None:
                shm_y.close(); shm_y.unlink()

        mi_matrix = pd.DataFrame(scores, index=feature_cols, columns=target_cols_agg)
        feature_cols_ranked = mi_matrix.mean(axis=1).sort_values(ascending=False)
        return feature_cols_ranked, mi_matrix

    feature_cols_ranked, mi_matrix = _mutual_information_scoring()

    df = pd.DataFrame({
        "rank": range(1, len(feature_cols_ranked) + 1),
        "feature": feature_cols_ranked.index,
        "mean_mi": feature_cols_ranked.values,
        "target": os.environ["TARGET"],
        "feature_dataset": Path(os.environ["FEATURE_DATASET"]).stem,
    }).set_index("feature")

    df_horizons = mi_matrix.reindex(feature_cols_ranked.index)
    feature_data = pd.concat([df, df_horizons], axis=1).reset_index(names="feature")

    display(feature_data[:3])
    return feature_data

feature_data = rank_features(
    features=features,
    future_prediction_targets_agg=future_prediction_targets_agg,
    feature_selection_subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_START"]),
    feature_selection_subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_SUBSAMPLE_END"]),
    feature_selection_subsample_amount=int(os.environ["FEATURE_RANKING_SUBSAMPLE_AMOUNT"]),
    output_resolution_minutes=int(os.environ["OUTPUT_RESOLUTION"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
feature_data.to_parquet(
    resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_feature_data_{_stem}.parquet")
)


Loading features parquet: \\wsl.localhost\Ubuntu\home\daniel\Projects\Forecasting\3_Features_select\..\2_Features_build\Feature_data\1_dispatch_price.parquet
  loaded features: shape=(736417, 634) in 29.8s
Loading aggregated targets parquet for target=NSW, dataset=1_dispatch_price
  loaded targets: shape=(736417, 96) in 4.7s
  filtering to subsample window [2019-01-01 00:00:00 .. 2023-01-01 00:00:00]
    window rows: features=420,769 targets=420,769 shared=420,769
  detecting discrete features
    discrete features: 92 / 634 in 4.0s
  preparing feature ndarray (shape=(420769, 634))
    ready in 1.3s | feature matrix 1018MB | contiguous=True
  subsampling 1,000 of 420,769 rows
  MI scoring: 634 features × 96 horizons (30-min) | subsample n=1,000 | 192 tasks (2 feature chunks × 96 buckets) across 7 CPUs | ~923MB per worker
  allocating shared memory: X=2MB, y=0MB
    shared memory ready in 0.0s
  spawning 7 loky workers and dispatching 192 tasks (first results may take ~30s while workers

[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
MI scoring:   0%|          | 0/192 [00:00<?, ?it/s]

    first worker result received after 10.7s


MI scoring: 100%|██████████| 192/192 [02:13<00:00,  1.44it/s]


,feature,rank,mean_mi,target,feature_dataset,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32,h33,h34,h35,h36,h37,h38,h39,h40,h41,h42,h43,h44,h45,h46,h47,h48,h49,h50,h51,h52,h53,h54,h55,h56,h57,h58,h59,h60,h61,h62,h63,h64,h65,h66,h67,h68,h69,h70,h71,h72,h73,h74,h75,h76,h77,h78,h79,h80,h81,h82,h83,h84,h85,h86,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_q90_336,1,0.520164,NSW,1_dispatch_price,0.638863,0.585473,0.606601,0.618337,0.583480,0.576743,0.570995,0.564746,0.557964,0.526674,0.554107,0.529595,0.523742,0.461112,0.530932,0.534999,0.534638,0.528452,0.600258,0.539052,0.550476,0.614291,0.549602,0.608067,0.596580,0.505441,0.507992,0.528501,0.471504,0.484849,0.461433,0.452710,0.475150,0.472977,0.507453,0.559380,0.560127,0.550470,0.559854,0.567691,0.607022,0.586470,0.605131,0.553087,0.584957,0.579249,0.594688,0.553951,0.492338,0.552416,0.562416,0.534253,0.467108,0.486600,0.452145,0.485226,0.489191,0.428339,0.445461,0.479580,0.465433,0.459163,0.464526,0.435891,0.418751,0.500343,0.500606,0.491494,0.556279,0.507189,0.469493,0.475734,0.505722,0.481861,0.453238,0.438423,0.417968,0.418797,0.482438,0.472270,0.476713,0.452088,0.535210,0.518749,0.579973,0.531870,0.486889,0.488713,0.505644,0.537211,0.515098,0.538254,0.535173,0.513770,0.508212,0.505639
1,nsw_price_asinh_rmean_336,2,0.508457,NSW,1_dispatch_price,0.669668,0.601920,0.612534,0.636281,0.627400,0.601567,0.624224,0.540811,0.537632,0.523939,0.526626,0.527934,0.514671,0.525278,0.517664,0.516195,0.486623,0.531660,0.556299,0.523825,0.540229,0.512724,0.565422,0.523272,0.551111,0.512797,0.484811,0.512637,0.470028,0.464717,0.450288,0.460500,0.489705,0.546980,0.546631,0.575655,0.524799,0.536079,0.551488,0.537154,0.582216,0.593880,0.580602,0.556457,0.543631,0.570113,0.535440,0.577510,0.475813,0.491969,0.485420,0.506332,0.470341,0.498873,0.474476,0.459533,0.452788,0.479713,0.437834,0.421766,0.426041,0.451613,0.457722,0.462982,0.475821,0.492980,0.487037,0.483651,0.539279,0.481138,0.457238,0.489295,0.422522,0.492058,0.445113,0.442738,0.428636,0.399256,0.420610,0.463200,0.456595,0.436282,0.497134,0.475553,0.483733,0.448766,0.499841,0.500438,0.471376,0.495269,0.492289,0.545606,0.504920,0.498430,0.510044,0.522200
2,nsw_price_rmean_2016,3,0.496087,NSW,1_dispatch_price,0.527689,0.566931,0.505186,0.549046,0.533929,0.547354,0.498999,0.514184,0.565817,0.535580,0.503777,0.527859,0.483552,0.513701,0.499600,0.539287,0.548703,0.513125,0.533715,0.574611,0.527763,0.513766,0.524695,0.505945,0.508257,0.507750,0.471431,0.508754,0.497336,0.495440,0.452167,0.401541,0.453020,0.511102,0.453084,0.491286,0.521304,0.485608,0.501109,0.468756,0.525776,0.540875,0.527598,0.474595,0.534144,0.486363,0.509003,0.469705,0.433704,0.443740,0.477701,0.517582,0.453206,0.534580,0.483386,0.472560,0.472733,0.459443,0.428083,0.497421,0.447383,0.485964,0.456664,0.471321,0.496966,0.544277,0.481334,0.512906,0.527601,0.497474,0.456383,0.532066,0.475015,0.458292,0.459867,0.447893,0.506253,0.442617,0.475158,0.490255,0.442541,0.500104,0.500882,0.495819,0.490455,0.519124,0.523606,0.492460,0.485441,0.473860,0.505556,0.482530,0.471714,0.476842,0.479530,0.493238
